In [1]:
!pip install kagglehub

In [2]:
import kagglehub
import pandas as pd
import os

pd.set_option('display.max_columns', None)

# Download dataset
path = kagglehub.dataset_download("tawfikelmetwally/employee-dataset")

print("Path dataset:", path)

# List contents of the downloaded directory to verify its structure
print("Contents of downloaded directory:", os.listdir(path))

# Load the single dataset file
df = pd.read_csv(os.path.join(path, "Employee.csv"))

display(df.info())

Using Colab cache for faster access to the 'employee-dataset' dataset.
Path dataset: /kaggle/input/employee-dataset
Contents of downloaded directory: ['Employee.csv']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4653 entries, 0 to 4652
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Education                  4653 non-null   object
 1   JoiningYear                4653 non-null   int64 
 2   City                       4653 non-null   object
 3   PaymentTier                4653 non-null   int64 
 4   Age                        4653 non-null   int64 
 5   Gender                     4653 non-null   object
 6   EverBenched                4653 non-null   object
 7   ExperienceInCurrentDomain  4653 non-null   int64 
 8   LeaveOrNot                 4653 non-null   int64 
dtypes: int64(5), object(4)
memory usage: 327.3+ KB


None

# **1. Identifikasi Teknik Pemodelan**

**Konteks**:

Data set ini berisi informasi tentang karyawan di sebuah perusahaan, termasuk latar belakang pendidikan, riwayat kerja, demografi, dan faktor-faktor terkait pekerjaan. Data ini telah dianonimkan untuk melindungi privasi sambil tetap memberikan wawasan berharga tentang tenaga kerja.

**Kolom:**

- *Education*: Kualifikasi pendidikan karyawan, termasuk gelar, institusi, dan bidang studi.
- *Joining Year*: Tahun ketika setiap karyawan bergabung dengan perusahaan, menunjukkan lama masa kerja mereka.
- City: Lokasi atau kota tempat setiap karyawan berdomisili atau bekerja.
- *Payment Tier*: Pengelompokan karyawan ke dalam tingkatan gaji yang berbeda.
- *Age*: Usia setiap karyawan, memberikan wawasan demografis.
- *Gender*: Identitas jenis kelamin karyawan, mendukung analisis keragaman.
- *Ever Benched*: Menunjukkan apakah seorang karyawan pernah sementara waktu tanpa tugas yang ditugaskan.
- *Experience in Current Domain*: Jumlah tahun pengalaman karyawan di bidang mereka saat ini.
- *Leave or Not*: kolom target


**Penggunaan:**

Data set ini dapat digunakan untuk berbagai analisis terkait sumber daya manusia (SDM) dan tenaga kerja, termasuk analisis retensi karyawan, evaluasi struktur gaji, studi keragaman dan inklusi, serta analisis pola cuti. Peneliti, analis data, dan profesional SDM dapat memperoleh wawasan berharga dari data set ini.

**Pertanyaan Penelitian Potensial:**

- Bagaimana distribusi kualifikasi pendidikan di antara karyawan?
- Bagaimana variasi lama masa kerja (Tahun Masuk) di berbagai kota?
- Apakah ada korelasi antara Tingkatan Gaji dan Pengalaman di Bidang Saat Ini?
- Bagaimana distribusi gender di dalam tenaga kerja?
- Apakah ada pola dalam perilaku pengambilan cuti di antara karyawan?

In [3]:
df.head()

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
0,Bachelors,2017,Bangalore,3,34,Male,No,0,0
1,Bachelors,2013,Pune,1,28,Female,No,3,1
2,Bachelors,2014,New Delhi,3,38,Female,No,2,0
3,Masters,2016,Bangalore,3,27,Male,No,5,1
4,Masters,2017,Pune,3,24,Male,Yes,2,1


In [4]:
# Numeric features
df.describe()

,JoiningYear,PaymentTier,Age,ExperienceInCurrentDomain,LeaveOrNot
count,4653.000000,4653.000000,4653.000000,4653.000000,4653.000000
mean,2015.062970,2.698259,29.393295,2.905652,0.343864
std,1.863377,0.561435,4.826087,1.558240,0.475047
min,2012.000000,1.000000,22.000000,0.000000,0.000000
25%,2013.000000,3.000000,26.000000,2.000000,0.000000
50%,2015.000000,3.000000,28.000000,3.000000,0.000000
75%,2017.000000,3.000000,32.000000,4.000000,1.000000
max,2018.000000,3.000000,41.000000,7.000000,1.000000


In [5]:
# Categorical features
df.describe(include=['object'])

,Education,City,Gender,EverBenched
count,4653,4653,4653,4653
unique,3,3,2,2
top,Bachelors,Bangalore,Male,No
freq,3601,2228,2778,4175


In [6]:
# Count missing values
df.isnull().sum().to_frame().T

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
0,0,0,0,0,0,0,0,0,0


In [7]:
# Unique values per column
df.nunique().to_frame().T

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
0,3,7,3,3,20,2,2,8,2


# **2. Pemilihan Teknik**

| Model                                               | Cocok Jika                                                                                                           | Kelebihan                                                                        | Kekurangan                                                   |
| --------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------- | ------------------------------------------------------------ |
| **Logistic Regression**                             | • Ingin model sederhana & mudah diinterpretasi<br>• Hubungan fitur–target relatif linear<br>• Dataset kecil–menengah | • Cepat<br>• Koefisien mudah dijelaskan<br>• Baseline yang sangat baik           | • Kurang bagus untuk pola non-linear kompleks                |
| **Decision Tree**                                   | • Hubungan data non-linear<br>• Membutuhkan model yang mudah dijelaskan secara visual                                | • Interpretabel<br>• Bisa menangani fitur numerik & kategorikal                  | • Mudah overfitting                                          |
| **Random Forest**                                   | • Menginginkan performa bagus tanpa banyak tuning<br>• Data tabular dengan interaksi kompleks                        | • Lebih stabil dari decision tree<br>• Tahan terhadap overfitting                | • Kurang interpretabel dibanding logistic regression         |
| **Gradient Boosting (XGBoost, LightGBM, CatBoost)** | • Mengejar akurasi tinggi<br>• Dataset menengah–besar                                                                | • Performa sangat kuat di data tabular<br>• Menangani non-linearitas dengan baik | • Perlu tuning<br>• Lebih kompleks                           |
| **Support Vector Machine (SVM)**                    | • Dataset kecil–menengah<br>• Dimensi fitur tinggi                                                                   | • Efektif di high-dimensional space                                              | • Kurang scalable untuk data besar<br>• Sulit diinterpretasi |
| **K-Nearest Neighbors (KNN)**                       | • Dataset kecil<br>• Pola keputusan lokal                                                                            | • Sederhana dan intuitif                                                         | • Lambat saat inferensi<br>• Sensitif terhadap skala fitur   |


# **3. Skenario Pengujian**

## **3.1. Menentukan Target**

In [8]:
X = df.drop(columns='LeaveOrNot')
y = df['LeaveOrNot']

print(X.shape)
print(y.shape)

(4653, 8)
(4653,)


## **3.2. Preprocessing**
- Encoding kolom kategorikal
- Scaling kolom numerik

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

num_features = X.select_dtypes(include='int64').columns
cat_features = X.select_dtypes(include='object').columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first'), cat_features)
    ]
)

print('Numeric features:', list(num_features))
print('Categorical features:', list(cat_features))

Numeric features: ['JoiningYear', 'PaymentTier', 'Age', 'ExperienceInCurrentDomain']
Categorical features: ['Education', 'City', 'Gender', 'EverBenched']


## **3.3 .Mendefinisikan Model & Scoring**

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(probability=True),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(
        eval_metric='logloss',
        random_state=42
    )
}

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

## **3.4. Cross Validation**

In [11]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []

for name, model in models.items():
    pipe = Pipeline([
        ('preprocess', preprocessor),
        ('model', model)
    ])

    cv_results = cross_validate(
        pipe,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results.append({
        'Model': name,
        'Accuracy (mean)': cv_results['test_accuracy'].mean(),
        'Precision (mean)': cv_results['test_precision'].mean(),
        'Recall (mean)': cv_results['test_recall'].mean(),
        'F1-score (mean)': cv_results['test_f1'].mean(),
        'ROC-AUC (mean)': cv_results['test_roc_auc'].mean()
    })

results_df = pd.DataFrame(results).sort_values(by='ROC-AUC (mean)', ascending=False)
results_df

,Model,Accuracy (mean),Precision (mean),Recall (mean),F1-score (mean),ROC-AUC (mean)
5,XGBoost,0.840102,0.836932,0.665625,0.741064,0.863564
4,Random Forest,0.828065,0.796075,0.673125,0.729234,0.852324
3,SVM,0.842683,0.876061,0.632500,0.734338,0.840095
2,KNN,0.811949,0.782986,0.626875,0.696223,0.824849
1,Decision Tree,0.810223,0.757137,0.661250,0.705701,0.787241
0,Logistic Regression,0.732864,0.682046,0.418750,0.518601,0.731498


# **4. Kesimpulan Praktikum**

## 1) Ringkasan performa (per poin penting)

* **XGBoost** — *Terbaik secara keseluruhan*:

  * **Accuracy 0.840**, **Precision 0.837**, **Recall 0.666**, **F1 0.741**, **ROC-AUC 0.864**.
  * ROC-AUC dan F1 tertinggi → model paling seimbang antara kemampuan memisah kelas dan harmoni precision/recall.
* **Random Forest** — *Alternatif kuat jika prioritas recall sedikit lebih tinggi*:

  * **Accuracy 0.828**, **Precision 0.796**, **Recall 0.673**, **F1 0.729**, **ROC-AUC 0.852**.
  * Recall sedikit **lebih tinggi** dari XGBoost (0.673 vs 0.666) — berguna kalau ingin sedikit lebih banyak menangkap karyawan yang akan resign.
* **SVM** — *Sangat selektif (precision tinggi)*:

  * **Precision 0.876** (tertinggi) tetapi **Recall 0.633** → sedikit banyak false negatives. Cocok kalau biaya false positive sangat tinggi.
* **KNN & Decision Tree** — performa menengah; mudah dipahami (tree) tapi kurang stabil dibanding ensemble.
* **Logistic Regression** — *baseline interpretabel tapi performa paling lemah*:

  * **Recall rendah (0.419)** → banyak karyawan yang akan resign tidak terdeteksi.

## 2) Interpretasi bisnis — trade-offs penting

* **Precision tinggi** (mis. SVM): prediksi “akan resign” lebih dapat dipercaya → **sedikit salah intervensi**. Cocok jika intervensi mahal.
* **Recall tinggi** (mis. RF sedikit lebih tinggi): lebih banyak resign yang *terdeteksi* → cocok jika goal adalah *mencegah* resign sebanyak mungkin, walau berarti intervensi lebih banyak.
* **XGBoost** menawarkan **keseimbangan terbaik** antara kedua sisi (F1 & ROC-AUC tertinggi), sehingga paling cocok bila Anda butuh model produksi yang andal.

## 3) Rekomendasi praktis (lanjutkan langkah ini)

1. **Pilih XGBoost sebagai model utama** (optimalkan ROC-AUC / F1).
2. **Gunakan Random Forest sebagai model cadangan** atau untuk eksperimen jika Anda ingin memaksimalkan recall sedikit lebih tinggi.
3. **Lakukan hyperparameter tuning** (GridSearch / Optuna) dengan scoring sesuai prioritas (mis. `scoring='recall'` jika recall utama; atau `'roc_auc'`/`'f1'` untuk keseimbangan).
4. **Thresholding & Precision-Recall analysis**: jangan terpaku pada threshold 0.5 — sesuaikan threshold untuk trade-off precision/recall yang diinginkan.
5. **Kalibrasi probabilitas** (Platt scaling / isotonic) jika Anda memakai thresholding atau ingin probabilitas yang dapat diinterpretasi.
6. **Explainability**: jalankan **SHAP** untuk XGBoost/RF agar HR paham fitur apa yang memicu prediksi “will leave”.
7. **Cek class balance & sampling**: jika kelas `LeaveOrNot` imbalanced, pertimbangkan SMOTE / class_weight pada model.
8. **Monitoring**: setelah deploy, monitor ROC-AUC, precision/recall per minggu/bulan dan recalibrate jika distribusi berubah.


## 4) Kesimpulan

> “Dari evaluasi 5-fold stratified CV pada dataset ini, **XGBoost** menunjukkan performa terbaik secara umum (ROC-AUC 0.864, F1 0.741). **Random Forest** mendekati kinerja XGBoost dengan recall sedikit lebih tinggi (0.673) sehingga layak dipertimbangkan bila tujuan utama adalah memaksimalkan detection rate. **Logistic Regression** memberikan baseline interpretabel namun recall yang rendah (0.419), sehingga tidak direkomendasikan sebagai model utama dalam konteks pencegahan resign.”